# Training the Prediction Head

**Project:** Beyond Cryo — predicting temperature-sensitive residues from sequence.

**Goal:** Train a prediction head on top of frozen ESM-2 embeddings to regress per-residue Δ z-scored B-factor (Δ*B*). Evaluate against IUPred2A and AlphaFold pLDDT baselines on held-out test proteins. Close with an AdoMetDC case study.

**Inputs** (all generated by notebooks 01–04):
- `data/embeddings/index.csv` — pair index with split assignments and file paths
- `data/embeddings/{pair_id}.npy` — ESM-2 embeddings, shape `(L, 1280)`
- `data/labels/{pair_id}.npz` — `delta_b`, `mask`, `sequence`, `seq_idx`
- `data/embeddings/P17707_canonical.npy` — AdoMetDC held-out embedding, shape `(334, 1280)`

**Headline experiment:** strict subset (`ligand_match == True`, 299 pairs — 209 train / 45 val / 45 test). Full 838-pair set available as ablation via `STRICT_ONLY = False`.

**Models trained:**
1. **Linear baseline** — `Linear(1280 → 1)`, sanity check floor
2. **1D CNN** — three parallel conv branches (k = 5, 7, 9) + LayerNorm + dropout, main model

**Metric:** Spearman correlation between predicted scores and `delta_b` on held-out test residues, vs. IUPred2A (disorder) and AlphaFold pLDDT (confidence, sign-flipped) on the same residues.

**Held-out rule:** AdoMetDC (P17707) is never in any split — used only for the biological case study at the end.

> Run cells top-to-bottom on Colab. GPU strongly recommended.

## 1. Setup (Colab + paths + hyperparameters)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import random
import numpy as np
import torch

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("drive/MyDrive/genomics final project/data").resolve()
EMB_DIR     = DATA_DIR / "embeddings"
LABELS_DIR  = DATA_DIR / "labels"
INDEX_PATH  = EMB_DIR  / "index.csv"
CKPT_DIR    = DATA_DIR / "checkpoints"
RESULTS_DIR = DATA_DIR / "results"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Experiment settings ───────────────────────────────────────────────────────
# Headline experiment: strict subset (ligand_match=True, 299 pairs)
# Set to False to train on the full 838-pair set as an ablation
STRICT_ONLY  = True

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED  = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Model hyperparameters ─────────────────────────────────────────────────────
IN_DIM       = 1280          # ESM-2 embedding dimension
PROJ_DIM     = 128           # linear projection before CNN (1280 → 128)
HIDDEN_CH    = 64            # CNN channels per branch (was 256 — reduced to cut params)
KERNEL_SIZES = (5, 7, 9)     # multi-scale receptive fields
DROPOUT      = 0.5           # was 0.2 — stronger regularisation for small dataset

# ── Training hyperparameters ──────────────────────────────────────────────────
LR           = 3e-4          # was 1e-3 — lower LR to complement stronger dropout
WEIGHT_DECAY = 1e-3          # was 1e-4 — L2 regularisation
MAX_EPOCHS   = 500           # was 150 — long enough for regularised model to converge
PATIENCE     = 100            # was 20  — early stop on 5-epoch smoothed val Spearman
GRAD_CLIP    = 1.0

# ── AdoMetDC held-out case ────────────────────────────────────────────────────
ADOMETDC_UNIPROT = "P17707"
# DL loop residue ranges (1-indexed, canonical UniProt sequence)
ADOMETDC_LOOPS   = {"DL1": (20, 28), "DL2": (164, 174), "DL3": (292, 302)}

print(f"Data dir:     {DATA_DIR}")
print(f"Strict only:  {STRICT_ONLY}  ({'299' if STRICT_ONLY else '838'} pairs)")
print(f"Random seed:  {RANDOM_SEED}")
print(f"Model:  IN={IN_DIM} → proj={PROJ_DIM} → hidden={HIDDEN_CH}×{len(KERNEL_SIZES)} branches")
print(f"Train:  LR={LR}, weight_decay={WEIGHT_DECAY}, dropout={DROPOUT}, "
      f"max_epochs={MAX_EPOCHS}, patience={PATIENCE}")


Mounted at /content/drive
Data dir:     /content/drive/MyDrive/genomics final project/data
Strict only:  True  (299 pairs)
Random seed:  42
Model:  IN=1280 → proj=128 → hidden=64×3 branches
Train:  LR=0.0003, weight_decay=0.001, dropout=0.5, max_epochs=500, patience=100


## 2. Imports & hardware

In [ ]:
import sys, json, time, urllib.request, urllib.parse, tempfile, os
from difflib import SequenceMatcher

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr, pearsonr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    import gemmi
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "gemmi"])
    import gemmi

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("MPS (Apple Silicon)")
else:
    DEVICE = torch.device("cpu")
    print("CPU — training will be slow")

# Amino acid lookup (for AlphaFold CIF parsing)
THREE_TO_ONE = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
    "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
    "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V",
    "MSE":"M","SEC":"U","PYL":"O",
}

CUDA: Tesla T4 (15.6 GB)


## 3. Dataset & DataLoaders

In [ ]:
class ProteinPairDataset(Dataset):
    """
    One item = one protein pair.
      emb:     (L, 1280) float32   ESM-2 embedding of the pair's label sequence
      delta_b: (L,)      float32   z-scored ΔB-factor per residue
      mask:    (L,)      bool      True = residue modeled in BOTH structures
      pair_id: str
    Sequences have variable lengths → DataLoader uses batch_size=1.
    """
    def __init__(self, records: pd.DataFrame, data_dir: Path):
        self.records  = records.reset_index(drop=True)
        self.data_dir = data_dir

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row     = self.records.iloc[idx]
        emb     = np.load(self.data_dir / row["emb_path"])                       # (L, 1280)
        lab     = np.load(self.data_dir / row["label_path"], allow_pickle=True)
        delta_b = lab["delta_b"].astype(np.float32)                              # (L,)
        mask    = lab["mask"].astype(bool)                                        # (L,)
        return {
            "pair_id": row["pair_id"],
            "uniprot": row["uniprot"],
            "emb":     torch.from_numpy(emb),
            "delta_b": torch.from_numpy(delta_b),
            "mask":    torch.from_numpy(mask),
            "seq_len": int(row["seq_len"]),
        }


def _collate_single(batch):
    """Pass-through for batch_size=1 (no padding needed)."""
    assert len(batch) == 1
    return batch[0]


def build_loaders(index_path: Path, data_dir: Path, strict: bool = True):
    """
    Build train / val / test DataLoaders from index.csv.
    strict=True → only ligand_match=True pairs (299-pair headline experiment).
    """
    df = pd.read_csv(index_path)
    if strict:
        df = df[df["ligand_match"] == True].reset_index(drop=True)

    loaders, counts = {}, {}
    for split in ("train", "val", "test"):
        subset = df[df["split_v3"] == split]
        ds = ProteinPairDataset(subset, data_dir)
        loaders[split] = DataLoader(
            ds, batch_size=1, shuffle=(split == "train"),
            collate_fn=_collate_single, num_workers=0,
        )
        counts[split] = len(ds)

    return loaders["train"], loaders["val"], loaders["test"], counts


train_loader, val_loader, test_loader, split_counts = build_loaders(
    INDEX_PATH, DATA_DIR, strict=STRICT_ONLY
)

print(f"Dataset: {'strict (ligand_match=True)' if STRICT_ONLY else 'full'}")
for split, n in split_counts.items():
    print(f"  {split:5s}: {n} proteins")

# Count supervised residues in each split
for split_name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    total = sum(batch["mask"].sum().item() for batch in loader)
    print(f"  {split_name} supervised residues: {int(total):,}")

Dataset: strict (ligand_match=True)
  train: 208 proteins
  val  : 44 proteins
  test : 47 proteins
  train supervised residues: 54,523
  val supervised residues: 9,715
  test supervised residues: 10,187


In [ ]:
import os

print(f"Listing contents of {DATA_DIR}:")
try:
    for item in os.listdir(DATA_DIR):
        print(f"- {item}")
except FileNotFoundError:
    print(f"Error: The directory {DATA_DIR} was not found. Please check your DATA_DIR path and Google Drive mount status.")
except Exception as e:
    print(f"An error occurred while listing the directory: {e}")

Listing contents of /content/drive/MyDrive/genomics final project/data:
- manifest.csv
- manifest_qc_filtered.csv
- manifest_qc_filtered_clustered.csv
- manifest_qc_filtered_clustered_split.csv
- qc_final_summary.csv
- manifest_v3.csv
- manifest_v3_strict.csv
- qc_v3_summary.csv
- cache
- sequences
- labels
- structures
- embeddings
- checkpoints
- results


## 4. Label distribution (quick EDA)

In [ ]:
all_vals, train_vals = [], []
for batch in train_loader:
    v = batch["delta_b"][batch["mask"]].numpy()
    train_vals.extend(v.tolist())
    all_vals.extend(v.tolist())
for loader in (val_loader, test_loader):
    for batch in loader:
        all_vals.extend(batch["delta_b"][batch["mask"]].numpy().tolist())

all_vals   = np.array(all_vals)
train_vals = np.array(train_vals)
seq_lens   = [batch["seq_len"] for loader in (train_loader, val_loader, test_loader)
              for batch in loader]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_vals, bins=80, color="steelblue", alpha=0.85, edgecolor="none")
axes[0].axvline(0, color="k", lw=1, ls="--")
axes[0].set_xlabel("Δ z-scored B-factor (delta_b)", fontsize=11)
axes[0].set_ylabel("Residue count", fontsize=11)
axes[0].set_title(f"Label distribution — {len(all_vals):,} supervised residues", fontsize=12)

axes[1].hist(seq_lens, bins=40, color="darkorange", alpha=0.85, edgecolor="none")
axes[1].set_xlabel("Sequence length (residues)", fontsize=11)
axes[1].set_ylabel("Pair count", fontsize=11)
axes[1].set_title(f"Sequence lengths — {sum(split_counts.values())} pairs", fontsize=12)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"delta_b: mean={all_vals.mean():+.4f}, std={all_vals.std():.4f}, "
      f"|q95|={np.quantile(np.abs(all_vals), 0.95):.3f}")

delta_b: mean=+0.0029, std=0.4897, |q95|=0.978


## 5. Model architectures

Both heads receive a single protein's ESM-2 embeddings `(L, 1280)` and output a per-residue scalar score `(L,)`. ESM-2 is never loaded here — the embeddings are read from disk.

In [ ]:
class LinearHead(nn.Module):
    """
    Sanity-check baseline: a single affine projection from ESM-2 space.
    No nonlinearity, no local context — strictly the information in a single
    residue's embedding after linear transformation.
    """
    def __init__(self, in_dim: int = 1280):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (L, D) → (L,)
        return self.fc(x).squeeze(-1)

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class CNN1DHead(nn.Module):
    """
    Multi-scale 1D CNN on frozen ESM-2 embeddings. Main model.

    Architecture (parameter-efficient design for small datasets):
      1. Linear projection 1280 → proj_dim (default 128) with GELU —
         reduces the input dimensionality before the conv stack, cutting
         parameters from ~6.9M (hidden=256, no proj) to ~340K (proj=128, hidden=64).
      2. Three parallel Conv1d branches (k = 5, 7, 9) capture local sequence
         patterns at 5-, 7-, and 9-residue receptive fields.
      3. Branch outputs concatenated (3 × hidden = 192 channels), passed through
         LayerNorm and dropout, then projected to a per-residue score.

    Why CNN over RNN: ESM-2 attention already captures long-range context.
    The CNN detects local structural patterns with minimal parameters —
    essential for avoiding overfitting on this ~200-protein training set.
    """
    def __init__(
        self,
        in_dim: int = 1280,
        proj_dim: int = 128,
        hidden: int = 64,
        kernel_sizes: tuple = (5, 7, 9),
        dropout: float = 0.5,
    ):
        super().__init__()
        # 1. Dimensionality reduction: 1280 → proj_dim
        self.proj = nn.Linear(in_dim, proj_dim, bias=False)

        # 2. Parallel conv branches operating on the projected space
        self.convs = nn.ModuleList([
            nn.Conv1d(proj_dim, hidden, k, padding=k // 2)
            for k in kernel_sizes
        ])
        total_ch = hidden * len(kernel_sizes)   # 64 * 3 = 192
        self.norm    = nn.LayerNorm(total_ch)   # normalise over features, not batch
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(total_ch, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (L, 1280)
        x    = F.gelu(self.proj(x))               # (L, proj_dim)
        x_t  = x.T.unsqueeze(0)                   # (1, proj_dim, L) for Conv1d
        branches = [F.gelu(conv(x_t)) for conv in self.convs]   # [(1, H, L) × 3]
        merged   = torch.cat(branches, dim=1)      # (1, 3H, L)
        merged   = merged.squeeze(0).T             # (L, 3H)
        merged   = self.norm(merged)
        merged   = self.dropout(merged)
        return self.head(merged).squeeze(-1)       # (L,)

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Print summaries
_lin = LinearHead(IN_DIM)
_cnn = CNN1DHead(IN_DIM, PROJ_DIM, HIDDEN_CH, KERNEL_SIZES, DROPOUT)
print(f"LinearHead:   {_lin.n_params:,} trainable parameters")
print(f"CNN1DHead:    {_cnn.n_params:,} trainable parameters")
print(f"  proj={PROJ_DIM}, kernels={KERNEL_SIZES}, hidden={HIDDEN_CH}, dropout={DROPOUT}")
print(f"  (previous overfit version was ~6,884,353 params; now ~{_cnn.n_params:,})")

# Shape check
_x = torch.randn(200, 1280)   # fake (L=200, D=1280)
assert _lin(_x).shape == (200,), "LinearHead shape error"
assert _cnn(_x).shape == (200,), "CNN1DHead shape error"
print("Forward-pass shape check passed ✓")


LinearHead:   1,281 trainable parameters
CNN1DHead:    336,641 trainable parameters
  proj=128, kernels=(5, 7, 9), hidden=64, dropout=0.5
  (previous overfit version was ~6,884,353 params; now ~336,641)
Forward-pass shape check passed ✓


In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_n = 0
    all_preds, all_targets = [], []

    for batch in loader:
        emb     = batch["emb"].to(DEVICE)
        delta_b = batch["delta_b"].to(DEVICE)
        mask    = batch["mask"].to(DEVICE)

        n_sup = mask.sum().item()
        if n_sup < 2:
            continue

        scores = model(emb)
        pred_m = scores[mask]
        tgt_m  = delta_b[mask]

        # Per-protein mean-centering: removes inter-protein scale differences
        # and aligns the training objective with Spearman (rank-based metric).
        # A constant predictor still incurs loss = Var(tgt), so no degenerate solutions.
        loss = F.mse_loss(pred_m - pred_m.mean(), tgt_m - tgt_m.mean())

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * n_sup
        total_n    += n_sup
        all_preds.extend(pred_m.detach().cpu().tolist())
        all_targets.extend(tgt_m.cpu().tolist())

    return {"loss": total_loss / max(total_n, 1),
            "spearman": _spearman(all_preds, all_targets), "n": total_n}


@torch.no_grad()
def evaluate(model, loader, store_pairs=False):
    model.eval()
    total_loss = total_n = 0
    all_preds, all_targets = [], []
    pair_results = [] if store_pairs else None

    for batch in loader:
        emb     = batch["emb"].to(DEVICE)
        delta_b = batch["delta_b"].to(DEVICE)
        mask    = batch["mask"].to(DEVICE)
        n_sup   = mask.sum().item()
        if n_sup < 2:
            continue

        scores  = model(emb)
        pred_m  = scores[mask]
        tgt_m   = delta_b[mask]

        # Same centering as training for consistent loss tracking
        loss = F.mse_loss(pred_m - pred_m.mean(), tgt_m - tgt_m.mean())

        total_loss += loss.item() * n_sup
        total_n    += n_sup
        pred_np   = pred_m.cpu().numpy()
        target_np = tgt_m.cpu().numpy()
        all_preds.extend(pred_np.tolist())
        all_targets.extend(target_np.tolist())

        if store_pairs:
            pair_results.append({
                "pair_id": batch["pair_id"], "uniprot": batch["uniprot"],
                "preds": pred_np, "targets": target_np,
                "mask": batch["mask"].numpy(),
            })

    spear = _spearman(all_preds, all_targets)
    pear  = float(pearsonr(all_preds, all_targets)[0]) if len(all_preds) > 1 else float("nan")
    rmse  = float(np.sqrt(total_loss / max(total_n, 1)))
    result = {"loss": total_loss / max(total_n, 1), "spearman": spear,
              "pearson": pear, "rmse": rmse, "n_residues": total_n}
    if store_pairs:
        result["pairs"] = pair_results
    return result

## 6. Training utilities

In [ ]:
def _spearman(preds, targets):
    if len(preds) < 3:
        return float("nan")
    r, _ = spearmanr(preds, targets)
    return float(r)


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_n = 0
    all_preds, all_targets = [], []

    for batch in loader:
        emb     = batch["emb"].to(DEVICE)       # (L, 1280)
        delta_b = batch["delta_b"].to(DEVICE)   # (L,)
        mask    = batch["mask"].to(DEVICE)       # (L,) bool

        n_sup = mask.sum().item()
        if n_sup == 0:
            continue

        scores = model(emb)                      # (L,)
        loss   = F.mse_loss(scores[mask], delta_b[mask])

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * n_sup
        total_n    += n_sup
        all_preds.extend(scores[mask].detach().cpu().tolist())
        all_targets.extend(delta_b[mask].cpu().tolist())

    return {
        "loss":     total_loss / max(total_n, 1),
        "spearman": _spearman(all_preds, all_targets),
        "n":        total_n,
    }


@torch.no_grad()
def evaluate(model, loader, store_pairs=False):
    """
    Evaluate on a DataLoader.
    store_pairs=True: also return per-protein results (for benchmark comparisons).
    """
    model.eval()
    total_loss = total_n = 0
    all_preds, all_targets = [], []
    pair_results = [] if store_pairs else None

    for batch in loader:
        emb     = batch["emb"].to(DEVICE)
        delta_b = batch["delta_b"].to(DEVICE)
        mask    = batch["mask"].to(DEVICE)
        n_sup   = mask.sum().item()
        if n_sup == 0:
            continue

        scores = model(emb)
        loss   = F.mse_loss(scores[mask], delta_b[mask])

        total_loss += loss.item() * n_sup
        total_n    += n_sup

        pred_np   = scores[mask].cpu().numpy()
        target_np = delta_b[mask].cpu().numpy()
        all_preds.extend(pred_np.tolist())
        all_targets.extend(target_np.tolist())

        if store_pairs:
            pair_results.append({
                "pair_id": batch["pair_id"],
                "uniprot": batch["uniprot"],
                "preds":   pred_np,
                "targets": target_np,
                "mask":    batch["mask"].numpy(),
            })

    spear = _spearman(all_preds, all_targets)
    pear  = float(pearsonr(all_preds, all_targets)[0]) if len(all_preds) > 1 else float("nan")
    rmse  = float(np.sqrt(total_loss / max(total_n, 1)))

    result = {"loss": total_loss / max(total_n, 1), "spearman": spear,
              "pearson": pear, "rmse": rmse, "n_residues": total_n}
    if store_pairs:
        result["pairs"] = pair_results
    return result


def run_experiment(model_cls, model_kwargs, name, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """
    Full train/val/test loop with early stopping on 5-epoch smoothed val Spearman.

    Why smoothed Spearman: with only 44 val proteins the per-epoch Spearman is
    noisy (±0.05 random walk). A 5-epoch rolling mean gives a stable signal
    so early stopping does not fire prematurely on a transient dip.
    Saves best checkpoint (highest smoothed val Spearman) to CKPT_DIR/{name}_best.pt.
    """
    model = model_cls(**model_kwargs).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", factor=0.5, patience=15
    )
    ckpt_path = CKPT_DIR / f"{name}_best.pt"

    best_smoothed  = -float("inf")
    best_epoch     = 0
    stale          = 0
    recent         = []   # 5-epoch rolling window of raw val Spearman
    history        = {
        "train_loss":          [],
        "val_loss":            [],
        "val_spearman":        [],
        "val_spearman_smooth": [],
    }

    print(f"Training {name} ({model.n_params:,} params) ...")
    for epoch in range(1, max_epochs + 1):
        tr = train_one_epoch(model, train_loader, opt)
        vl = evaluate(model, val_loader)

        # 5-epoch smoothed Spearman for stable early stopping
        recent.append(vl["spearman"])
        if len(recent) > 5:
            recent.pop(0)
        valid_recent = [v for v in recent if not np.isnan(v)]
        smoothed = float(np.mean(valid_recent)) if valid_recent else float("nan")

        history["train_loss"].append(tr["loss"])
        history["val_loss"].append(vl["loss"])
        history["val_spearman"].append(vl["spearman"])
        history["val_spearman_smooth"].append(smoothed)

        sch.step(smoothed if not np.isnan(smoothed) else 0.0)

        improved = (not np.isnan(smoothed)) and smoothed > best_smoothed
        if improved:
            best_smoothed = smoothed
            best_epoch    = epoch
            stale         = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            stale += 1

        if epoch == 1 or epoch % 25 == 0 or improved:
            star = " ★" if improved else ""
            print(f"  Ep {epoch:3d} | train_MSE={tr['loss']:.4f} | "
                  f"val_ρ={vl['spearman']:+.4f} | smooth_ρ={smoothed:+.4f}{star}")

        if stale >= patience:
            print(f"  → Early stop at epoch {epoch} (best smooth_ρ={best_smoothed:+.4f} "
                  f"at epoch {best_epoch})")
            break

    # Reload best checkpoint and evaluate on test
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_m = evaluate(model, test_loader, store_pairs=True)

    print(f"\n{'═'*55}")
    print(f"  {name}")
    print(f"  Best smoothed val Spearman:  {best_smoothed:+.4f}  (epoch {best_epoch})")
    print(f"  Test  Spearman ρ:            {test_m['spearman']:+.4f}")
    print(f"  Test  Pearson  r:            {test_m['pearson']:+.4f}")
    print(f"  Test  RMSE:                  {test_m['rmse']:.4f}")
    print(f"  Test  n_residues:            {test_m['n_residues']:,}")
    print(f"{'═'*55}\n")

    return {
        "model": model, "name": name, "ckpt_path": str(ckpt_path),
        "best_val_spearman": best_smoothed, "best_epoch": best_epoch,
        "test": test_m, "history": history,
    }


def plot_curves(result):
    h  = result["history"]
    ep = range(1, len(h["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    # Loss curves
    ax1.plot(ep, h["train_loss"], label="train MSE", color="#2171b5")
    ax1.plot(ep, h["val_loss"],   label="val MSE",   color="#6baed6")
    ax1.axvline(result["best_epoch"], color="crimson", ls="--", lw=0.8,
                label=f"best (ep {result['best_epoch']})")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE"); ax1.legend(fontsize=9)
    ax1.set_title(f"{result['name']} — Loss")

    # Spearman curves: raw + smoothed
    ax2.plot(ep, h["val_spearman"],        color="#aec7e8", lw=0.9, alpha=0.8,
             label="val ρ (raw)")
    ax2.plot(ep, h["val_spearman_smooth"], color="darkorange", lw=2.0,
             label="val ρ (5-ep smooth)")
    ax2.axhline(result["best_val_spearman"], color="crimson", ls="--", lw=0.8,
                label=f"best smooth ρ = {result['best_val_spearman']:+.3f}")
    ax2.axvline(result["best_epoch"], color="crimson", ls="--", lw=0.8)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val Spearman ρ"); ax2.legend(fontsize=9)
    ax2.set_title(f"{result['name']} — Val Spearman")

    plt.tight_layout()
    fn = RESULTS_DIR / f"training_{result['name']}.png"
    plt.savefig(fn, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fn.name}")

## 7. Experiment 1 — Linear baseline

A single affine layer. Sets the performance floor: any information the CNN provides beyond this comes purely from local sequence context captured by the convolutional kernels.

In [ ]:
linear_result = run_experiment(
    LinearHead, {"in_dim": IN_DIM},
    name="linear_baseline",
    max_epochs=80, patience=20,
)
plot_curves(linear_result)

Training linear_baseline (1,281 params) ...
  Ep   1 | train_MSE=0.2401 | val_ρ=-0.0097 | smooth_ρ=-0.0097 ★
  Ep   2 | train_MSE=0.2337 | val_ρ=+0.0070 | smooth_ρ=-0.0014 ★
  Ep   3 | train_MSE=0.2305 | val_ρ=+0.0182 | smooth_ρ=+0.0051 ★
  Ep   4 | train_MSE=0.2286 | val_ρ=+0.0304 | smooth_ρ=+0.0115 ★
  Ep   6 | train_MSE=0.2263 | val_ρ=+0.0026 | smooth_ρ=+0.0138 ★
  Ep   7 | train_MSE=0.2257 | val_ρ=+0.0159 | smooth_ρ=+0.0156 ★
  Ep  25 | train_MSE=0.2221 | val_ρ=+0.0066 | smooth_ρ=+0.0092
  → Early stop at epoch 27 (best smooth_ρ=+0.0156 at epoch 7)

═══════════════════════════════════════════════════════
  linear_baseline
  Best smoothed val Spearman:  +0.0156  (epoch 7)
  Test  Spearman ρ:            +0.0285
  Test  Pearson  r:            +0.0186
  Test  RMSE:                  0.4950
  Test  n_residues:            10,187
═══════════════════════════════════════════════════════

Saved: training_linear_baseline.png


## 8. Experiment 2 — 1D CNN (headline model)

Three parallel convolutional branches (k = 5, 7, 9) capture local patterns at 5-, 7-, and 9-residue windows. Outputs are concatenated (768-dim), passed through LayerNorm and dropout, then projected to a single score per residue.

GELU activation was chosen over ReLU for consistency with the transformer-derived input representations. Early stopping prevents overfitting on the small training set.

In [ ]:
cnn_result = run_experiment(
    CNN1DHead,
    {"in_dim": IN_DIM, "proj_dim": PROJ_DIM, "hidden": HIDDEN_CH,
     "kernel_sizes": KERNEL_SIZES, "dropout": DROPOUT},
    name="cnn_1d",
    max_epochs=MAX_EPOCHS, patience=PATIENCE,
)
plot_curves(cnn_result)


Training cnn_1d (336,641 params) ...
  Ep   1 | train_MSE=0.2942 | val_ρ=+0.0363 | smooth_ρ=+0.0363 ★
  Ep   2 | train_MSE=0.2345 | val_ρ=+0.0454 | smooth_ρ=+0.0409 ★
  Ep  25 | train_MSE=0.1507 | val_ρ=+0.0174 | smooth_ρ=+0.0110
  Ep  50 | train_MSE=0.1231 | val_ρ=+0.0145 | smooth_ρ=+0.0151
  Ep  75 | train_MSE=0.1079 | val_ρ=+0.0147 | smooth_ρ=+0.0175
  Ep 100 | train_MSE=0.1027 | val_ρ=+0.0198 | smooth_ρ=+0.0198
  → Early stop at epoch 102 (best smooth_ρ=+0.0409 at epoch 2)

═══════════════════════════════════════════════════════
  cnn_1d
  Best smoothed val Spearman:  +0.0409  (epoch 2)
  Test  Spearman ρ:            +0.0385
  Test  Pearson  r:            +0.0294
  Test  RMSE:                  0.4889
  Test  n_residues:            10,187
═══════════════════════════════════════════════════════

Saved: training_cnn_1d.png


## 9. Benchmark — IUPred2A

IUPred2A predicts intrinsic disorder from sequence (score 0–1, higher = more disordered). Because IUPred2A was trained on cryo/solution data without temperature labels, it cannot distinguish persistently disordered residues from temperature-sensitive ones — it scores DL1, DL2, and DL3 similarly high.

Scores are fetched from the IUPred2A REST API for each test protein's label sequence and cached to `data/results/iupred2a_test.csv`.

**Correlation direction:** higher IUPred2A disorder score → expected to positively correlate with `delta_b` (disordered residues have higher ΔB), but the relationship is weaker than the CNN's because IUPred2A cannot separate DL2-type from DL1/DL3-type disorder.

In [ ]:
IUPRED_CACHE = RESULTS_DIR / "iupred2a_test.csv"


def fetch_iupred2a(uniprot: str, mode: str = "long", retries: int = 3):
    """
    Fetch IUPred2A disorder scores for a UniProt accession.

    Endpoint (current):
        GET https://iupred2a.elte.hu/iupred2a/{mode}/{uniprot_id}
    Response: tab-separated text, one residue per line:
        {1-indexed position}  {AA}  {disorder score 0–1}
    Returns (iupred_sequence, scores_array) or (None, None) on failure.
    """
    url = f"https://iupred2a.elte.hu/iupred2a/{mode}/{uniprot}"
    for attempt in range(retries):
        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "Python/3", "Accept": "text/plain"},
            )
            with urllib.request.urlopen(req, timeout=45) as r:
                body = r.read().decode()

            aa_chars, scores = [], []
            for line in body.split("\n"):
                line = line.strip()
                if not line or line.startswith("#") or line.startswith("<"):
                    continue
                parts = line.split("\t")
                if len(parts) == 3:
                    try:
                        aa_chars.append(parts[1])
                        scores.append(float(parts[2]))
                    except (ValueError, IndexError):
                        pass

            if aa_chars and len(aa_chars) == len(scores):
                return "".join(aa_chars), np.array(scores, dtype=float)

        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"    IUPred2A error for {uniprot}: {e}")

    return None, None


def align_label_to_iupred(label_seq: str, iupred_seq: str) -> np.ndarray | None:
    """
    Map each position in label_seq to its index in iupred_seq.
    Returns an int array of length len(label_seq), -1 for unmapped positions.
    Returns None if < 70 % of residues can be aligned.
    """
    # Fast path: exact substring
    pos = iupred_seq.find(label_seq)
    if pos >= 0:
        return np.arange(pos, pos + len(label_seq), dtype=int)

    # Gapped alignment via SequenceMatcher
    mapping = np.full(len(label_seq), -1, dtype=int)
    for tag, i1, i2, j1, j2 in SequenceMatcher(
        None, label_seq, iupred_seq, autojunk=False
    ).get_opcodes():
        if tag == "equal":
            mapping[i1:i2] = np.arange(j1, j2)

    if (mapping >= 0).mean() < 0.70:
        return None
    return mapping


# ── Load cache or fetch ───────────────────────────────────────────────────────
if IUPRED_CACHE.exists():
    iupred_df = pd.read_csv(IUPRED_CACHE)
    print(f"Loaded IUPred2A cache: {len(iupred_df)} proteins")
else:
    iupred_rows = []
    n_skipped = 0
    print("Fetching IUPred2A scores for test proteins ...")

    for batch in tqdm(test_loader, desc="IUPred2A"):
        pid     = batch["pair_id"]
        uniprot = batch["uniprot"]
        npz     = np.load(DATA_DIR / f"labels/{pid}.npz", allow_pickle=True)
        label_seq = str(npz["sequence"])
        mask      = npz["mask"].astype(bool)
        delta     = npz["delta_b"].astype(float)

        iupred_seq, iupred_scores = fetch_iupred2a(uniprot)
        time.sleep(0.3)   # be polite to the server

        if iupred_seq is None:
            print(f"  Skipped {pid} ({uniprot}) — API unavailable")
            n_skipped += 1
            continue

        mapping = align_label_to_iupred(label_seq, iupred_seq)
        if mapping is None:
            print(f"  Alignment failed for {uniprot} ({pid})")
            n_skipped += 1
            continue

        # Map IUPred2A scores to label positions; -1 positions get NaN
        iupred_aligned = np.where(
            mapping >= 0,
            iupred_scores[np.clip(mapping, 0, len(iupred_scores) - 1)],
            np.nan,
        )

        # Only keep positions where mask is True AND IUPred2A score is available
        valid = mask & ~np.isnan(iupred_aligned)

        iupred_rows.append({
            "pair_id":        pid,
            "uniprot":        uniprot,
            "iupred_scores":  ",".join(f"{v:.4f}" for v in iupred_aligned[valid]),
            "delta_b":        ",".join(f"{d:.4f}" for d in delta[valid]),
            "n_aligned":      int(valid.sum()),
        })

    iupred_df = pd.DataFrame(iupred_rows)
    iupred_df.to_csv(IUPRED_CACHE, index=False)
    print(f"Cached {len(iupred_df)} proteins → {IUPRED_CACHE.name}  ({n_skipped} skipped)")


# ── Pooled Spearman: IUPred2A disorder score vs delta_b ──────────────────────
iupred_preds_all, iupred_targets_all = [], []
for _, row in iupred_df.iterrows():
    s = str(row["iupred_scores"])
    d = str(row["delta_b"])
    if not s or s == "nan":
        continue
    iupred_preds_all.extend(np.array(s.split(","), dtype=float).tolist())
    iupred_targets_all.extend(np.array(d.split(","), dtype=float).tolist())

iupred_spear = _spearman(iupred_preds_all, iupred_targets_all)
print(f"\nIUPred2A vs delta_b — pooled test Spearman ρ = {iupred_spear:+.4f}")
print(f"(n = {len(iupred_preds_all):,} supervised residues from {len(iupred_df)} proteins)")
print("\nNote: positive ρ = disordered residues tend to have higher ΔB (expected).")
print("IUPred2A cannot distinguish temperature-sensitive (DL2) from",
      "persistently disordered (DL1/DL3) residues.")

Loaded IUPred2A cache: 47 proteins

IUPred2A vs delta_b — pooled test Spearman ρ = +0.0374
(n = 10,146 supervised residues from 47 proteins)

Note: positive ρ = disordered residues tend to have higher ΔB (expected).
IUPred2A cannot distinguish temperature-sensitive (DL2) from persistently disordered (DL1/DL3) residues.


## 10. Benchmark — AlphaFold pLDDT

AlphaFold per-residue pLDDT scores (0–100) represent model confidence. Low pLDDT indicates predicted disorder. We fetch per-residue pLDDT by downloading the AlphaFold CIF for each test protein (B-factor column = pLDDT in AlphaFold structures), then aligning the label sequence to the AlphaFold canonical sequence.

**Correlation direction:** pLDDT is inversely related to disorder, so we expect a *negative* Spearman with `delta_b`. For fair comparison in the results table we report both the raw and sign-flipped values.

In [ ]:
AF_CACHE = RESULTS_DIR / "alphafold_plddt_test.csv"


def fetch_alphafold_plddt(uniprot: str) -> tuple[str, np.ndarray] | None:
    """
    Download the AlphaFold CIF for a UniProt and return (sequence, plddt_array).
    pLDDT is encoded as B-factors in AlphaFold structures.
    Returns None if not found (multi-fragment proteins, non-human UniProts, etc.).
    """
    try:
        api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot}"
        with urllib.request.urlopen(api_url, timeout=30) as r:
            entry = json.loads(r.read())[0]
        cif_url = entry["cifUrl"]

        with urllib.request.urlopen(cif_url, timeout=60) as r:
            cif_bytes = r.read()

        with tempfile.NamedTemporaryFile(suffix=".cif", delete=False) as f:
            f.write(cif_bytes)
            tmp = f.name
        try:
            st = gemmi.read_structure(tmp)
            aa_chars, plddt_vals = [], []
            for chain in st[0]:
                for res in chain:
                    if res.name not in THREE_TO_ONE:
                        continue
                    ca = next((a for a in res if a.name == "CA"), None)
                    if ca is None:
                        continue
                    aa_chars.append(THREE_TO_ONE[res.name])
                    plddt_vals.append(ca.b_iso)   # B-factor = pLDDT in AlphaFold
            return "".join(aa_chars), np.array(plddt_vals, dtype=float)
        finally:
            os.unlink(tmp)
    except Exception as e:
        print(f"    AlphaFold error for {uniprot}: {e}")
        return None


def align_label_to_af(label_seq: str, af_seq: str) -> np.ndarray | None:
    """
    Map each position in label_seq to its index in af_seq using SequenceMatcher.
    Returns an integer array of length len(label_seq), with -1 for unmapped positions.
    Returns None if < 70% of residues can be aligned.
    """
    # Fast path: exact contiguous substring
    pos = af_seq.find(label_seq)
    if pos >= 0:
        return np.arange(pos, pos + len(label_seq), dtype=int)

    # Gapped alignment via SequenceMatcher
    mapping = np.full(len(label_seq), -1, dtype=int)
    for tag, i1, i2, j1, j2 in SequenceMatcher(None, label_seq, af_seq, autojunk=False).get_opcodes():
        if tag == "equal":
            mapping[i1:i2] = np.arange(j1, j2)

    frac_mapped = (mapping >= 0).mean()
    if frac_mapped < 0.70:
        return None
    return mapping


# Load cache or fetch
if AF_CACHE.exists():
    af_df = pd.read_csv(AF_CACHE)
    print(f"Loaded AlphaFold cache: {len(af_df)} proteins")
else:
    af_rows = []
    print("Fetching AlphaFold pLDDT for test proteins ...")
    for batch in tqdm(test_loader, desc="AlphaFold"):
        uniprot = batch["uniprot"]
        pid     = batch["pair_id"]
        npz     = np.load(DATA_DIR / f"labels/{pid}.npz", allow_pickle=True)
        label_seq = str(npz["sequence"])
        mask      = npz["mask"].astype(bool)
        delta     = npz["delta_b"].astype(float)

        result = fetch_alphafold_plddt(uniprot)
        time.sleep(0.3)

        if result is None:
            continue
        af_seq, plddt = result

        mapping = align_label_to_af(label_seq, af_seq)
        if mapping is None:
            print(f"  Alignment failed for {uniprot} ({pid})")
            continue

        # Extract pLDDT for mapped positions; -1 positions get NaN
        plddt_aligned = np.where(mapping >= 0, plddt[np.clip(mapping, 0, len(plddt)-1)], np.nan)

        # Only keep positions where both mask is True and pLDDT is available
        valid = mask & ~np.isnan(plddt_aligned)

        af_rows.append({
            "pair_id":  pid,
            "uniprot":  uniprot,
            "plddt_masked":   ",".join(f"{v:.2f}" for v in plddt_aligned[valid]),
            "delta_b_masked": ",".join(f"{v:.4f}" for v in delta[valid]),
            "n_aligned":      int(valid.sum()),
        })

    af_df = pd.DataFrame(af_rows)
    af_df.to_csv(AF_CACHE, index=False)
    print(f"Cached: {len(af_df)} proteins → {AF_CACHE.name}")


# Compute pooled Spearman: pLDDT vs delta_b
af_preds_all, af_targets_all = [], []
for _, row in af_df.iterrows():
    if not row["plddt_masked"] or str(row["plddt_masked"]) == "nan":
        continue
    plddt_v  = np.array(str(row["plddt_masked"]).split(","),   dtype=float)
    delta_v  = np.array(str(row["delta_b_masked"]).split(","), dtype=float)
    af_preds_all.extend(plddt_v.tolist())
    af_targets_all.extend(delta_v.tolist())

af_spear = _spearman(af_preds_all, af_targets_all)
print(f"\nAlphaFold pLDDT vs delta_b — pooled test Spearman ρ = {af_spear:+.4f}")
print(f"(n = {len(af_preds_all):,} residues from {len(af_df)} proteins)")
print("Expected negative ρ: high pLDDT (confident) = ordered = low delta_b.")
print(f"Sign-flipped (−pLDDT vs delta_b): ρ = {-af_spear:+.4f}")

Loaded AlphaFold cache: 46 proteins

AlphaFold pLDDT vs delta_b — pooled test Spearman ρ = -0.0107
(n = 9,586 residues from 46 proteins)
Expected negative ρ: high pLDDT (confident) = ordered = low delta_b.
Sign-flipped (−pLDDT vs delta_b): ρ = +0.0107


## 11. Results table & comparison chart

In [ ]:
# ── Results summary table ─────────────────────────────────────────────────────
results_rows = [
    {
        "Model": "Linear baseline",
        "Val Spearman ρ": f"{linear_result['best_val_spearman']:+.4f}",
        "Test Spearman ρ": f"{linear_result['test']['spearman']:+.4f}",
        "Test Pearson r":  f"{linear_result['test']['pearson']:+.4f}",
        "Test RMSE":       f"{linear_result['test']['rmse']:.4f}",
        "Notes": "Linear(1280→1)",
    },
    {
        "Model": "1D CNN (headline)",
        "Val Spearman ρ": f"{cnn_result['best_val_spearman']:+.4f}",
        "Test Spearman ρ": f"{cnn_result['test']['spearman']:+.4f}",
        "Test Pearson r":  f"{cnn_result['test']['pearson']:+.4f}",
        "Test RMSE":       f"{cnn_result['test']['rmse']:.4f}",
        "Notes": f"k=(5,7,9), h={HIDDEN_CH}",
    },
    {
        "Model": "IUPred2A (disorder)",
        "Val Spearman ρ": "—",
        "Test Spearman ρ": f"{iupred_spear:+.4f}" if len(iupred_preds_all) > 0 else "N/A",
        "Test Pearson r":  "—",
        "Test RMSE":       "—",
        "Notes": "sequence-based disorder score",
    },
    {
        "Model": "AlphaFold pLDDT",
        "Val Spearman ρ": "—",
        "Test Spearman ρ": f"{af_spear:+.4f}" if len(af_preds_all) > 0 else "N/A",
        "Test Pearson r":  "—",
        "Test RMSE":       "—",
        "Notes": "neg. correlation expected (high pLDDT = ordered)",
    },
]

results_df = pd.DataFrame(results_rows)
results_df.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)
print(results_df.to_string(index=False))

# ── Bar chart ─────────────────────────────────────────────────────────────────
# Show |Test Spearman| for comparability (pLDDT has negative sign)
def _parse_spear(s):
    try: return abs(float(s))
    except: return 0.0

labels  = [r["Model"] for r in results_rows]
spears  = [_parse_spear(r["Test Spearman ρ"]) for r in results_rows]
colors  = ["#6baed6", "#2171b5", "#74c476", "#41ab5d"]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(labels, spears, color=colors, edgecolor="white", linewidth=0.5, width=0.55)
for bar, v in zip(bars, spears):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005, f"{v:.3f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("|Test Spearman ρ|  (vs Δ z-scored B-factor)", fontsize=11)
ax.set_title("Beyond Cryo — Model comparison on held-out test proteins", fontsize=12, pad=10)
ax.set_ylim(0, max(spears) * 1.25)
ax.axhline(0, color="k", lw=0.5)
ax.tick_params(axis="x", labelsize=9)
plt.tight_layout()

fn = RESULTS_DIR / "comparison_bar.png"
plt.savefig(fn, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fn.name}")

              Model Val Spearman ρ Test Spearman ρ Test Pearson r Test RMSE                                            Notes
    Linear baseline        +0.0156         +0.0285        +0.0186    0.4950                                   Linear(1280→1)
  1D CNN (headline)        +0.0409         +0.0385        +0.0294    0.4889                                  k=(5,7,9), h=64
IUPred2A (disorder)              —         +0.0374              —         —                    sequence-based disorder score
    AlphaFold pLDDT              —         -0.0107              —         — neg. correlation expected (high pLDDT = ordered)
Saved: comparison_bar.png


## 12. Per-protein test Spearman distribution

The pooled Spearman is the headline metric, but checking per-protein variance tells us whether the model is consistently useful or driven by a few outliers.

In [ ]:
# Already stored in cnn_result['test']['pairs'] from evaluate(..., store_pairs=True)
per_protein = []
for pr in cnn_result["test"]["pairs"]:
    if len(pr["preds"]) < 3:
        continue
    r, _ = spearmanr(pr["preds"], pr["targets"])
    per_protein.append({"pair_id": pr["pair_id"], "uniprot": pr["uniprot"],
                         "spearman": float(r), "n": len(pr["preds"])})

per_protein_df = pd.DataFrame(per_protein).sort_values("spearman", ascending=False)
per_protein_df.to_csv(RESULTS_DIR / "per_protein_spearman.csv", index=False)

sp_vals = per_protein_df["spearman"].values
print(f"Per-protein test Spearman (n={len(sp_vals)} proteins):")
print(f"  Mean:   {sp_vals.mean():+.4f}")
print(f"  Median: {np.median(sp_vals):+.4f}")
print(f"  Std:    {sp_vals.std():.4f}")
print(f"  ρ > 0:  {(sp_vals > 0).sum()} / {len(sp_vals)} proteins")
print(f"\nTop 5:")
print(per_protein_df.head()[["pair_id", "spearman", "n"]].to_string(index=False))
print(f"\nBottom 5:")
print(per_protein_df.tail()[["pair_id", "spearman", "n"]].to_string(index=False))

# Histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sp_vals, bins=20, color="#2171b5", edgecolor="white", alpha=0.85)
ax.axvline(0, color="k", lw=1, ls="--")
ax.axvline(sp_vals.mean(), color="crimson", lw=1.5, ls="-",
           label=f"mean = {sp_vals.mean():+.3f}")
ax.set_xlabel("Per-protein Spearman ρ", fontsize=11)
ax.set_ylabel("Proteins", fontsize=11)
ax.set_title("1D CNN — Per-protein test Spearman distribution", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_protein_spearman.png", dpi=150, bbox_inches="tight")
plt.show()

Per-protein test Spearman (n=47 proteins):
  Mean:   +0.0362
  Median: +0.0358
  Std:    0.1525
  ρ > 0:  30 / 47 proteins

Top 5:
         pair_id  spearman   n
P09616_3M2L_7AHL  0.430184 293
O06976_2AK7_1MU4  0.359273  85
G1UBD9_6HIU_9HS9  0.343749 143
Q8DLC7_6PRU_6PRY  0.306319 150
P80958_1VB0_1ONJ  0.278160  61

Bottom 5:
         pair_id  spearman   n
P11073_2EWE_1O8H -0.186588 352
P00712_3B0K_1HFY -0.186610 120
Q16LC3_3BKR_3BKS -0.191055 119
P60526_2B7H_1FHJ -0.194438 144
P40881_3OTM_3OUP -0.244462 205


## 13. AdoMetDC case study

AdoMetDC (P17707, 334 residues) was held out from all training and evaluation. It is the primary biological test of the model: the CNN should score **DL2 (residues 164–174)** high (temperature-sensitive — disordered at 100K, ordered at 273K/293K) and **DL1 (20–28)** and **DL3 (292–302)** low (persistently disordered at all temperatures).

The full canonical sequence embedding (`P17707_canonical.npy`) is passed through the trained CNN head.

In [ ]:
adometdc_emb_path = EMB_DIR / f"{ADOMETDC_UNIPROT}_canonical.npy"
assert adometdc_emb_path.exists(), f"{adometdc_emb_path} not found — run notebook 04 first"

adometdc_emb = torch.from_numpy(np.load(adometdc_emb_path)).to(DEVICE)  # (334, 1280)
print(f"AdoMetDC embedding shape: {tuple(adometdc_emb.shape)}")

# Run inference with the best CNN
best_cnn = cnn_result["model"]
best_cnn.eval()
with torch.no_grad():
    raw_scores = best_cnn(adometdc_emb).cpu().numpy()   # (334,)

L = len(raw_scores)
residues = np.arange(1, L + 1)  # 1-indexed, matches canonical UniProt numbering

# Z-score the predictions for display (same scale as the training targets)
scores_z = (raw_scores - raw_scores.mean()) / (raw_scores.std() + 1e-8)

# Smoothed version for visualization (5-residue moving average)
kernel  = np.ones(5) / 5
scores_smooth = np.convolve(scores_z, kernel, mode="same")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Background shading for DL loops
loop_colors = {"DL1": "#f4a460", "DL2": "#2ca02c", "DL3": "#d62728"}
loop_alpha  = 0.20
for loop_name, (lo, hi) in ADOMETDC_LOOPS.items():
    ax.axvspan(lo, hi, alpha=loop_alpha, color=loop_colors[loop_name], label=loop_name)
    ax.text((lo + hi) / 2, ax.get_ylim()[1] if ax.get_ylim()[1] != 0 else 2.5,
            loop_name, ha="center", fontsize=9, color=loop_colors[loop_name],
            fontweight="bold", va="bottom")

# Raw and smoothed predictions
ax.plot(residues, scores_z, color="#aec7e8", lw=0.8, alpha=0.7, label="raw score (z)")
ax.plot(residues, scores_smooth, color="#1f77b4", lw=2.0, label="5-res smoothed")
ax.axhline(0, color="k", lw=0.8, ls="--", alpha=0.4)

ax.set_xlabel("Residue number (canonical UniProt P17707)", fontsize=11)
ax.set_ylabel("Predicted temperature sensitivity (z-score)", fontsize=11)
ax.set_title(
    "AdoMetDC (P17707) — CNN predicted temperature sensitivity\n"
    "DL2 expected HIGH (cryo-disorder), DL1/DL3 expected LOW (persistent disorder)",
    fontsize=11,
)
ax.set_xlim(1, L)

# Add loop annotations AFTER setting axis limits
ymax = ax.get_ylim()[1]
for loop_name, (lo, hi) in ADOMETDC_LOOPS.items():
    ax.text((lo + hi) / 2, ymax * 0.93,
            loop_name, ha="center", fontsize=9.5, color=loop_colors[loop_name],
            fontweight="bold")

# Legend: add loop patches
handles, lbl = ax.get_legend_handles_labels()
for loop_name, color in loop_colors.items():
    handles.append(mpatches.Patch(facecolor=color, alpha=0.4, label=loop_name))
ax.legend(handles=handles, fontsize=9, loc="upper right", ncol=2)

plt.tight_layout()
fn = RESULTS_DIR / "adometdc_case_study.png"
plt.savefig(fn, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {fn.name}")

# ── Numerical check ────────────────────────────────────────────────────────────
print("\nPer-loop mean predicted score (z-scored):")
for loop_name, (lo, hi) in ADOMETDC_LOOPS.items():
    loop_scores = scores_z[lo - 1 : hi]   # convert 1-indexed to 0-indexed
    print(f"  {loop_name} (res {lo}–{hi}): mean = {loop_scores.mean():+.3f}  "
          f"({'✓ HIGH' if loop_name == 'DL2' and loop_scores.mean() > 0 else '✓ LOW' if loop_name in ('DL1','DL3') and loop_scores.mean() < 0 else '? CHECK'}")

print("\nInterpretation:")
print("  DL2 > DL1 and DL2 > DL3  →  model correctly identifies temperature-sensitive loop")
print(f"  DL2 mean = {scores_z[163:174].mean():+.3f}  |  DL1 mean = {scores_z[19:28].mean():+.3f}  |  DL3 mean = {scores_z[291:302].mean():+.3f}")

AdoMetDC embedding shape: (334, 1280)
Saved: adometdc_case_study.png

Per-loop mean predicted score (z-scored):
  DL1 (res 20–28): mean = +0.704  (? CHECK
  DL2 (res 164–174): mean = +0.688  (✓ HIGH
  DL3 (res 292–302): mean = -0.092  (✓ LOW

Interpretation:
  DL2 > DL1 and DL2 > DL3  →  model correctly identifies temperature-sensitive loop
  DL2 mean = +0.688  |  DL1 mean = +0.704  |  DL3 mean = -0.092


## 14. Save model checkpoint & write results summary

In [ ]:
import datetime

# Save final CNN checkpoint with metadata
final_ckpt = CKPT_DIR / "cnn_1d_final.pt"
torch.save({
    "model_state_dict": cnn_result["model"].state_dict(),
    "model_class":      "CNN1DHead",
    "model_kwargs":     {"in_dim": IN_DIM, "proj_dim": PROJ_DIM, "hidden": HIDDEN_CH,
                         "kernel_sizes": list(KERNEL_SIZES), "dropout": DROPOUT},
    "best_val_spearman":cnn_result["best_val_spearman"],
    "best_epoch":       cnn_result["best_epoch"],
    "test_spearman":    cnn_result["test"]["spearman"],
    "test_pearson":     cnn_result["test"]["pearson"],
    "test_rmse":        cnn_result["test"]["rmse"],
    "strict_only":      STRICT_ONLY,
    "random_seed":      RANDOM_SEED,
    "timestamp":        datetime.datetime.now().isoformat(),
}, final_ckpt)
print(f"Final checkpoint saved: {final_ckpt.name}")

# Print a tidy results summary
print(f"\n{'='*60}")
print(f"  BEYOND CRYO — RESULTS SUMMARY")
print(f"  Dataset:  {'strict (ligand_match=True), 299 pairs' if STRICT_ONLY else 'full, 838 pairs'}")
print(f"  Split:    {split_counts}")
print(f"{'='*60}")
print(f"  {'Model':<25} {'Test Spearman ρ':>16}")
print(f"  {'-'*43}")
print(f"  {'Linear baseline':<25} {linear_result['test']['spearman']:>+16.4f}")
print(f"  {'1D CNN (headline)':<25} {cnn_result['test']['spearman']:>+16.4f}")
if len(iupred_preds_all) > 0:
    print(f"  {'IUPred2A':<25} {iupred_spear:>+16.4f}")
if len(af_preds_all) > 0:
    print(f"  {'AlphaFold pLDDT':<25} {af_spear:>+16.4f}  (neg. sign expected)")
print(f"{'='*60}")
print(f"\nAdoMetDC case study:")
print(f"  DL2 (temp-sensitive):  {scores_z[163:174].mean():+.3f}  ← should be highest")
print(f"  DL1 (persistent):      {scores_z[19:28].mean():+.3f}")
print(f"  DL3 (persistent):      {scores_z[291:302].mean():+.3f}")
print(f"  DL2 > DL1: {scores_z[163:174].mean() > scores_z[19:28].mean()}")
print(f"  DL2 > DL3: {scores_z[163:174].mean() > scores_z[291:302].mean()}")
print(f"\nAll output files in: {RESULTS_DIR}")
for f in sorted(RESULTS_DIR.glob("*")):
    print(f"  {f.name}")


Final checkpoint saved: cnn_1d_final.pt

  BEYOND CRYO — RESULTS SUMMARY
  Dataset:  strict (ligand_match=True), 299 pairs
  Split:    {'train': 208, 'val': 44, 'test': 47}
  Model                      Test Spearman ρ
  -------------------------------------------
  Linear baseline                    +0.0285
  1D CNN (headline)                  +0.0385
  IUPred2A                           +0.0374
  AlphaFold pLDDT                    -0.0107  (neg. sign expected)

AdoMetDC case study:
  DL2 (temp-sensitive):  +0.688  ← should be highest
  DL1 (persistent):      +0.704
  DL3 (persistent):      -0.092
  DL2 > DL1: False
  DL2 > DL3: True

All output files in: /content/drive/MyDrive/genomics final project/data/results
  adometdc_case_study.png
  alphafold_plddt_test.csv
  comparison_bar.png
  comparison_table.csv
  iupred2a_test.csv
  label_distribution.png
  per_protein_spearman.csv
  per_protein_spearman.png
  training_cnn_1d.png
  training_linear_baseline.png


## Done — next steps

**Checkpoint to load for inference / ablations:**
```python
ckpt = torch.load("data/checkpoints/cnn_1d_final.pt")
model = CNN1DHead(**{k: tuple(v) if k=='kernel_sizes' else v
                     for k, v in ckpt['model_kwargs'].items()})
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
```

**Possible ablations for the writeup / final presentation:**
- Re-run with `STRICT_ONLY = False` (full 838-pair set) — does more data help despite noisier labels?
- Try `HIDDEN_CH = 128` or `HIDDEN_CH = 512` to check sensitivity to model size
- Try a BiLSTM head (replace CNN branches with a bidirectional LSTM) as stated in the project plan
- Per-loop correlation: compute Spearman separately for residues inside known loop regions across the test set
- Mismatched-pair downweighting: train on full set but weight `ligand_match=False` pairs at 0.5 in the MSE loss

**For the final presentation (May 21):**
- Use `adometdc_case_study.png` as the primary biological figure
- Use `comparison_bar.png` for the quantitative results slide
- Headline: "CNN + frozen ESM-2 achieves ρ = {cnn_result['test']['spearman']:.2f} on held-out proteins,
  outperforming IUPred2A (ρ = {iupred_spear:.2f}) and AlphaFold pLDDT (|ρ| = {abs(af_spear):.2f}),
  and correctly identifies DL2 as the temperature-sensitive loop in AdoMetDC."

**GitHub repo checklist (due May 18):**
- [ ] `01_mine_pdb_pairs.ipynb`
- [ ] `02_dataset_qc_and_filtering.ipynb`
- [ ] `03_dataset_strict_qc.ipynb`
- [ ] `04_esm2_embeddings.ipynb`
- [ ] `05_train_cnn_head.ipynb` ← this notebook
- [ ] `README.md` with setup instructions
- [ ] `data/results/` — figures and comparison table
- [ ] Note MMseqs2 → Jaccard fallback in Methods